In [ ]:
import pandas as pd
import sqlite3

file_path = "/content/delivery_delay_root_cause_dataset_strong_v1.xlsx"

projects = pd.read_excel(file_path, sheet_name="Projects")
milestones = pd.read_excel(file_path, sheet_name="Milestones")
issues = pd.read_excel(file_path, sheet_name="Issues")
resources = pd.read_excel(file_path, sheet_name="Resources")
change_requests = pd.read_excel(file_path, sheet_name="ChangeRequests")

conn = sqlite3.connect("delivery_delay_project.db")

projects.to_sql("projects", conn, if_exists="replace", index=False)
milestones.to_sql("milestones", conn, if_exists="replace", index=False)
issues.to_sql("issues", conn, if_exists="replace", index=False)
resources.to_sql("resources", conn, if_exists="replace", index=False)
change_requests.to_sql("change_requests", conn, if_exists="replace", index=False)

# Drop tables if they already exist to avoid OperationalError
conn.execute("DROP TABLE IF EXISTS milestone_agg")
milestone_agg_query = """
CREATE TABLE milestone_agg AS
SELECT
    project_id,
    COUNT(*) AS milestone_count,
    SUM(CASE WHEN critical_flag = 1 THEN 1 ELSE 0 END) AS critical_milestone_count,
    SUM(CASE WHEN milestone_delay_days > 0 THEN 1 ELSE 0 END) AS delayed_milestone_count,
    SUM(CASE WHEN critical_flag = 1 AND milestone_delay_days > 0 THEN 1 ELSE 0 END) AS critical_milestone_missed_count,
    AVG(milestone_delay_days) AS avg_milestone_delay_days,
    MAX(milestone_delay_days) AS max_milestone_delay_days
FROM milestones
GROUP BY project_id
"""
conn.execute(milestone_agg_query)

conn.execute("DROP TABLE IF EXISTS issue_agg")
issue_agg_query = """
CREATE TABLE issue_agg AS
SELECT
    project_id,
    COUNT(*) AS issue_count,
    SUM(CASE WHEN resolved_flag = 0 THEN 1 ELSE 0 END) AS unresolved_issue_count,
    SUM(CASE WHEN severity = 'High' THEN 1 ELSE 0 END) AS high_severity_issue_count,
    AVG(days_open) AS avg_issue_days_open,
    SUM(CASE WHEN delay_category = 'Approval Delay' THEN 1 ELSE 0 END) AS approval_delay_issue_count,
    SUM(CASE WHEN delay_category = 'Vendor Delay' THEN 1 ELSE 0 END) AS vendor_delay_issue_count,
    SUM(CASE WHEN delay_category = 'Resource Shortage' THEN 1 ELSE 0 END) AS resource_shortage_issue_count,
    SUM(CASE WHEN delay_category = 'Scope Change' THEN 1 ELSE 0 END) AS scope_change_issue_count,
    SUM(CASE WHEN delay_category = 'Dependency Blocker' THEN 1 ELSE 0 END) AS dependency_blocker_issue_count,
    SUM(CASE WHEN delay_category = 'Procurement Delay' THEN 1 ELSE 0 END) AS procurement_delay_issue_count,
    SUM(CASE WHEN delay_category = 'Technical Issue' THEN 1 ELSE 0 END) AS technical_issue_count
FROM issues
GROUP BY project_id
"""
conn.execute(issue_agg_query)

conn.execute("DROP TABLE IF EXISTS project_analysis")
master_query = """
CREATE TABLE project_analysis AS
SELECT
    p.project_id,
    p.project_name,
    p.project_type,
    p.business_unit,
    p.region,
    p.priority,
    p.start_date,
    p.planned_end_date,
    p.actual_end_date,
    p.planned_duration_days,
    p.actual_duration_days,
    p.delay_days,
    p.delayed_flag,
    p.major_delay_flag,
    p.budget_gbp,
    p.sponsor,
    p.project_manager,

    m.milestone_count,
    m.critical_milestone_count,
    m.delayed_milestone_count,
    m.critical_milestone_missed_count,
    m.avg_milestone_delay_days,
    m.max_milestone_delay_days,

    i.issue_count,
    i.unresolved_issue_count,
    i.high_severity_issue_count,
    i.avg_issue_days_open,
    i.approval_delay_issue_count,
    i.vendor_delay_issue_count,
    i.resource_shortage_issue_count,
    i.scope_change_issue_count,
    i.dependency_blocker_issue_count,
    i.procurement_delay_issue_count,
    i.technical_issue_count,

    r.planned_team_size,
    r.actual_team_size,
    r.resource_gap,
    r.resource_gap_pct,
    r.overtime_hours,
    r.attrition_flag,

    c.change_request_count,
    c.approved_scope_change_count,
    c.late_scope_change_flag,
    c.scope_change_impact

FROM projects p
LEFT JOIN milestone_agg m
    ON p.project_id = m.project_id
LEFT JOIN issue_agg i
    ON p.project_id = i.project_id
LEFT JOIN resources r
    ON p.project_id = r.project_id
LEFT JOIN change_requests c
    ON p.project_id = c.project_id
"""
conn.execute(master_query)

project_analysis = pd.read_sql("SELECT * FROM project_analysis", conn)
project_analysis.head()

project_analysis["schedule_variance_pct"] = (
    project_analysis["delay_days"] / project_analysis["planned_duration_days"]
)

project_analysis["resource_gap_flag"] = (
    project_analysis["resource_gap_pct"] >= 0.15
).astype(int)

project_analysis["high_overtime_flag"] = (
    project_analysis["overtime_hours"] >= 80
).astype(int)

project_analysis["preventable_delay_flag"] = (
    (
        (project_analysis["approval_delay_issue_count"] > 0) |
        (project_analysis["vendor_delay_issue_count"] > 0) |
        (project_analysis["resource_shortage_issue_count"] > 0) |
        (project_analysis["scope_change_issue_count"] > 0)
    ).astype(int)
)

project_analysis["risk_score"] = (
    (project_analysis["unresolved_issue_count"] > 3).astype(int) * 2 +
    (project_analysis["high_severity_issue_count"] > 1).astype(int) * 2 +
    (project_analysis["delayed_milestone_count"] > 2).astype(int) * 2 +
    (project_analysis["late_scope_change_flag"] == 1).astype(int) * 2 +
    (project_analysis["resource_gap_flag"] == 1).astype(int) * 1 +
    (project_analysis["approval_delay_issue_count"] > 1).astype(int) * 1 +
    (project_analysis["vendor_delay_issue_count"] > 1).astype(int) * 1 +
    (project_analysis["high_overtime_flag"] == 1).astype(int) * 1
)

def risk_band(score):
    if score >= 6:
        return "High"
    elif score >= 3:
        return "Medium"
    else:
        return "Low"

project_analysis["risk_band"] = project_analysis["risk_score"].apply(risk_band)

project_analysis.to_csv("project_analysis_master_table.csv", index=False)
project_analysis.to_excel("project_analysis_master_table.xlsx", index=False)

# KPI Summary

In [ ]:
kpi_summary = {
    "total_projects": len(project_analysis),
    "delayed_projects": int(project_analysis["delayed_flag"].sum()),
    "delayed_project_pct": float(project_analysis["delayed_flag"].mean()),
    "major_delays": int(project_analysis["major_delay_flag"].sum()),
    "major_delay_pct": float(project_analysis["major_delay_flag"].mean()),
    "avg_delay_days": float(project_analysis["delay_days"].mean()),
    "median_delay_days": float(project_analysis["delay_days"].median()),
    "avg_schedule_variance_pct": float(project_analysis["schedule_variance_pct"].mean()),
    "high_risk_projects": int((project_analysis["risk_band"] == "High").sum()),
    "high_risk_project_pct": float((project_analysis["risk_band"] == "High").mean())
}

kpi_summary

delay_by_project_type = (
    project_analysis.groupby("project_type")
    .agg(
        total_projects=("project_id", "count"),
        delayed_projects=("delayed_flag", "sum"),
        avg_delay_days=("delay_days", "mean"),
        major_delay_pct=("major_delay_flag", "mean")
    )
    .reset_index()
)

delay_by_business_unit = (
    project_analysis.groupby("business_unit")
    .agg(
        total_projects=("project_id", "count"),
        delayed_projects=("delayed_flag", "sum"),
        avg_delay_days=("delay_days", "mean"),
        high_risk_projects=("risk_band", lambda x: (x == "High").sum())
    )
    .reset_index()
)

delay_by_region = (
    project_analysis.groupby("region")
    .agg(
        total_projects=("project_id", "count"),
        avg_delay_days=("delay_days", "mean"),
        delayed_project_pct=("delayed_flag", "mean")
    )
    .reset_index()
)

root_cause_summary = pd.DataFrame({
    "root_cause": [
        "Approval Delay",
        "Vendor Delay",
        "Resource Shortage",
        "Scope Change",
        "Dependency Blocker",
        "Procurement Delay",
        "Technical Issue"
    ],
    "issue_count": [
        project_analysis["approval_delay_issue_count"].sum(),
        project_analysis["vendor_delay_issue_count"].sum(),
        project_analysis["resource_shortage_issue_count"].sum(),
        project_analysis["scope_change_issue_count"].sum(),
        project_analysis["dependency_blocker_issue_count"].sum(),
        project_analysis["procurement_delay_issue_count"].sum(),
        project_analysis["technical_issue_count"].sum()
    ]
})

root_cause_summary = root_cause_summary.sort_values("issue_count", ascending=False)
root_cause_summary

root_cause_summary["issue_share_pct"] = (
    root_cause_summary["issue_count"] / root_cause_summary["issue_count"].sum()
)

cause_impact = pd.DataFrame({
    "root_cause": [
        "Approval Delay",
        "Vendor Delay",
        "Resource Shortage",
        "Scope Change",
        "Dependency Blocker",
        "Procurement Delay",
        "Technical Issue"
    ],
    "avg_delay_days_when_present": [
        project_analysis.loc[project_analysis["approval_delay_issue_count"] > 0, "delay_days"].mean(),
        project_analysis.loc[project_analysis["vendor_delay_issue_count"] > 0, "delay_days"].mean(),
        project_analysis.loc[project_analysis["resource_shortage_issue_count"] > 0, "delay_days"].mean(),
        project_analysis.loc[project_analysis["scope_change_issue_count"] > 0, "delay_days"].mean(),
        project_analysis.loc[project_analysis["dependency_blocker_issue_count"] > 0, "delay_days"].mean(),
        project_analysis.loc[project_analysis["procurement_delay_issue_count"] > 0, "delay_days"].mean(),
        project_analysis.loc[project_analysis["technical_issue_count"] > 0, "delay_days"].mean()
    ],
    "major_delay_pct_when_present": [
        project_analysis.loc[project_analysis["approval_delay_issue_count"] > 0, "major_delay_flag"].mean(),
        project_analysis.loc[project_analysis["vendor_delay_issue_count"] > 0, "major_delay_flag"].mean(),
        project_analysis.loc[project_analysis["resource_shortage_issue_count"] > 0, "major_delay_flag"].mean(),
        project_analysis.loc[project_analysis["scope_change_issue_count"] > 0, "major_delay_flag"].mean(),
        project_analysis.loc[project_analysis["dependency_blocker_issue_count"] > 0, "major_delay_flag"].mean(),
        project_analysis.loc[project_analysis["procurement_delay_issue_count"] > 0, "major_delay_flag"].mean(),
        project_analysis.loc[project_analysis["technical_issue_count"] > 0, "major_delay_flag"].mean()
    ]
})

cause_impact = cause_impact.sort_values("avg_delay_days_when_present", ascending=False)
cause_impact

phase_delay_summary = (
    milestones.groupby("phase")
    .agg(
        milestone_count=("milestone_id", "count"),
        avg_milestone_delay_days=("milestone_delay_days", "mean"),
        max_milestone_delay_days=("milestone_delay_days", "max"),
        delayed_milestones=("milestone_delay_days", lambda x: (x > 0).sum())
    )
    .reset_index()
    .sort_values("avg_milestone_delay_days", ascending=False)
)

phase_delay_summary

risk_summary = (
    project_analysis.groupby("risk_band")
    .agg(
        total_projects=("project_id", "count"),
        avg_delay_days=("delay_days", "mean"),
        major_delay_pct=("major_delay_flag", "mean"),
        avg_unresolved_issues=("unresolved_issue_count", "mean")
    )
    .reset_index()
)

risk_summary

high_risk_projects = (
    project_analysis.loc[project_analysis["risk_band"] == "High"]
    .sort_values(["risk_score", "delay_days"], ascending=[False, False])
    [["project_id", "project_name", "project_type", "business_unit", "delay_days", "risk_score", "unresolved_issue_count", "late_scope_change_flag"]]
)

high_risk_projects.head(15)

delay_by_project_type.to_csv("delay_by_project_type.csv", index=False)
delay_by_business_unit.to_csv("delay_by_business_unit.csv", index=False)
delay_by_region.to_csv("delay_by_region.csv", index=False)
root_cause_summary.to_csv("root_cause_summary.csv", index=False)
cause_impact.to_csv("cause_impact.csv", index=False)
phase_delay_summary.to_csv("phase_delay_summary.csv", index=False)
risk_summary.to_csv("risk_summary.csv", index=False)
high_risk_projects.to_csv("high_risk_projects.csv", index=False)